# OCC 看跌占比对 SOXX 五日下行风险的增量信息

固定假设：交易前已完成的 SOXX 期权看跌占比相对过去 22 个交易日的异常值，可能在价格、波动和股票成交量之外改善未来五日下行风险预测。原始数据来自 OCC 清算总量，不是买方开仓、不含原始发布时间。2025 训练、2026 复用历史一次诊断；信息门仅为下一阶段资格，非 SOXX 经济门。

In [ ]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root:
        raise RuntimeError("workspace root missing")
    root = root.parent
card = root / "research/occ_put_share_tail_information_20261022"
plan_path = card / "plan.json"
plan = json.loads(plan_path.read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"]
assert not (card / "run_state.json").exists(), "one fixed information attempt already started"
source = plan["source"]
source_card = root / source["card"]
assert sha(source_card / "summary.json") == source["summary_sha256"]
assert sha(source_card / "verification_manifest.json") == source["verification_manifest_sha256"]
assert sha(source_card / "source_qualification_result.json") == source["qualification_result_sha256"]
for path,digest in plan["provider"]["files_sha256"].items():
    assert sha(root / path) == digest, path
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.inventory_sales_soxx_h63_information import stationary_bootstrap_lower
qlib.init(provider_uri=str(root / plan["provider"]["uri"]),region="us",kernels=1)
objects,source_audit = load_verified_recorder_objects(source_card,
    expected_recorder_id=source["recorder_id"],artifacts=[source["artifact"]],
    expected_artifact_sha256={source["artifact"]:source["artifact_sha256"]},
    expected_manifest_sha256=source["verification_manifest_sha256"])
occ = objects[source["artifact"]]
assert len(occ)==986 and set(occ.symbol)=={"SOXX","SOXL"}
occ_soxx = occ.loc[occ.symbol.eq("SOXX")].copy()
occ_soxx["date"] = pd.to_datetime(occ_soxx.date)
assert occ_soxx.date.is_unique and occ_soxx.date.is_monotonic_increasing
assert occ_soxx.put_share.notna().all() and occ_soxx.put_share.between(0,1).all()
cal = pd.DatetimeIndex(D.calendar(start_time=plan["provider"]["read_bounds"][0],
                                     end_time=plan["provider"]["read_bounds"][1],freq="day")).normalize()
raw = D.features(["SOXX"],plan["provider"]["fields"],start_time=cal[0],end_time=cal[-1],freq="day").reset_index()
assert set(raw.instrument)=={"SOXX"}
raw["datetime"] = pd.to_datetime(raw.datetime).dt.normalize()
assert pd.DatetimeIndex(raw.datetime).equals(cal)
assert raw[["$open","$close","$volume"]].notna().all().all()
assert raw[["$open","$close","$volume"]].gt(0).all().all()
prices = raw.set_index("datetime")[["$open","$close","$volume"]].astype(float)
expected_occ_dates = cal[(cal>=occ_soxx.date.min()) & (cal<=occ_soxx.date.max())]
assert pd.DatetimeIndex(occ_soxx.date).equals(expected_occ_dates)
put_share = occ_soxx.set_index("date").put_share.reindex(cal)
close,opening,volume = prices["$close"],prices["$open"],prices["$volume"]
ret = close.pct_change(fill_method=None)
frame = pd.DataFrame(index=cal)
frame["r1_lag"] = ret.shift(1)
frame["r5_lag"] = close.shift(1)/close.shift(6)-1
frame["rv5_lag"] = ret.shift(1).rolling(5,min_periods=5).std(ddof=0)
frame["rv22_lag"] = ret.shift(1).rolling(22,min_periods=22).std(ddof=0)
frame["volume_surprise_lag"] = np.log(volume.shift(1)/volume.shift(2).rolling(22,min_periods=22).median())
frame["put_surprise_lag"] = put_share.shift(1)-put_share.shift(2).rolling(22,min_periods=22).median()
# Existing fixed H5 open-to-open tail semantics; future bars never enter predictors.
daily = opening.shift(-2)/opening.shift(-1)-1
forward = pd.concat([daily.shift(-j) for j in range(5)],axis=1)
frame["tail_label"] = ((1+forward).cumprod(axis=1)-1).min(axis=1).le(-.06).astype(float)
frame.loc[forward.isna().any(axis=1),"tail_label"] = np.nan
frame["mature_at"] = pd.Series(cal,index=cal).shift(-6)+pd.Timedelta("9h31min")
train_start,train_end = map(pd.Timestamp,plan["windows"]["train_signal"])
held_start,held_end = map(pd.Timestamp,plan["windows"]["held_reused_diagnostic"])
train = frame.loc[train_start:train_end].copy()
held = frame.loc[held_start:held_end].copy()
columns = plan["feature_formula"]["baseline"]
# The frozen names above describe these concrete columns in this exact order.
baseline = ["r1_lag","r5_lag","rv5_lag","rv22_lag","volume_surprise_lag"]
candidate = baseline+["put_surprise_lag"]
assert len(columns)==len(baseline)
assert len(train)>=plan["windows"]["minimum_train_rows"] and len(held)>=plan["windows"]["minimum_held_rows"]
assert train[baseline+["put_surprise_lag","tail_label","mature_at"]].notna().all().all()
assert held[baseline+["put_surprise_lag","tail_label","mature_at"]].notna().all().all()
assert train.mature_at.max()<held.index.min()
train_events=int(train.tail_label.sum());held_events=int(held.tail_label.sum())
assert train_events>=plan["windows"]["minimum_train_events"] and held_events>=plan["windows"]["minimum_held_events"]
assert train.tail_label.nunique()==held.tail_label.nunique()==2
native=frame.loc[train.index.union(held.index),candidate+["tail_label"]].copy()
native["instrument"]="SOXX";native.index.name="datetime";native=native.reset_index().set_index(["datetime","instrument"])
panel=pd.concat({"feature":native[candidate],"label":native[["tail_label"]]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={"all":(None,None)})
dataset.config(dump_all=True,recursive=True)
prepared=dataset.prepare("all",col_set=["feature","label"])
pd.testing.assert_frame_equal(prepared,panel)
train_index=pd.MultiIndex.from_product([train.index,["SOXX"]],names=["datetime","instrument"])
held_index=pd.MultiIndex.from_product([held.index,["SOXX"]],names=["datetime","instrument"])
y_train=prepared["label"].loc[train_index,"tail_label"].to_numpy(dtype=int)
y_held=prepared["label"].loc[held_index,"tail_label"].to_numpy(dtype=int)
models={};prediction={}
(card / "run_state.json").write_text(json.dumps({"status":"fits_started","plan_sha256":sha(plan_path),"source_loads":1,"D_calendar_calls":1,"D_features_calls":1,"DatasetH_builds":1})+chr(10))
for name,features in (("baseline",baseline),("candidate",candidate)):
    scaler=StandardScaler()
    x_train=scaler.fit_transform(prepared["feature"].loc[train_index,features].to_numpy(float))
    x_held=scaler.transform(prepared["feature"].loc[held_index,features].to_numpy(float))
    assert np.isfinite(x_train).all() and np.isfinite(x_held).all() and (scaler.var_>0).all()
    model=LogisticRegression(C=1.0,max_iter=1000,class_weight=None,solver="lbfgs")
    model.fit(x_train,y_train)
    models[name]={"scaler":scaler,"model":model,"features":features}
    prediction[name]=model.predict_proba(x_held)[:,1]
held_result=held[["tail_label","put_surprise_lag","mature_at"]].copy()
held_result["baseline_probability"]=prediction["baseline"]
held_result["candidate_probability"]=prediction["candidate"]
held_result["brier_improvement"]=(y_held-prediction["baseline"])**2-(y_held-prediction["candidate"])**2
assert np.isfinite(held_result.brier_improvement).all()
gate=plan["information_gate"];boot=gate["bootstrap"]
lower=stationary_bootstrap_lower(held_result.brier_improvement.to_numpy(),
    mean_block=boot["mean_block_sessions"],draws=boot["draws"],seed=boot["seed"],quantile=boot["lower_quantile"])
coefficient=float(models["candidate"]["model"].coef_[0,-1])
h1=held_result.loc[held_result.index.month<=6].brier_improvement
h2=held_result.loc[held_result.index.month>=7].brier_improvement
assert len(h1) and len(h2)
passed=bool(coefficient>0 and held_result.brier_improvement.mean()>0 and h1.mean()>0 and h2.mean()>0 and lower>0)
result={"status":"not_promoted_source_PIT_gap" if passed else "not_promoted_information_gate_failed",
        "train_rows":len(train),"held_rows":len(held),"train_events":train_events,"held_events":held_events,
        "candidate_put_share_coefficient":coefficient,
        "held_brier_baseline":float(np.mean((y_held-prediction["baseline"])**2)),
        "held_brier_candidate":float(np.mean((y_held-prediction["candidate"])**2)),
        "held_paired_brier_improvement":float(held_result.brier_improvement.mean()),
        "H1_2026_paired_improvement":float(h1.mean()),"H2_2026_paired_improvement":float(h2.mean()),
        "stationary_bootstrap_lower":lower,"information_gate_pass":passed,
        "historical_first_public_time_qualified":False,"historical_window_independent":False,
        "new_model_fits":2,"new_bootstraps":1,"new_native_accounts":0,"qualified_forward_days":0,
        "SOXX_goal_achieved":False,"trading_or_promotion_permission":False,
        "plan_sha256":sha(plan_path),"source_recorder_id":source["recorder_id"]}
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="inputs",plan=plan,source_audit=source_audit,
                     price_provider_identity=plan["provider"]["files_sha256"])
    rec.save_objects(artifact_path="sources",source_daily=occ_soxx)
    rec.save_objects(artifact_path="models",baseline=models["baseline"],candidate=models["candidate"])
    rec.save_objects(artifact_path="predictions",held=held_result,train_features=train[candidate+["tail_label"]])
    rec.save_objects(artifact_path="analysis",result=result)
    rec.log_metrics(train_rows=len(train),held_rows=len(held),held_events=held_events,
                    held_brier_improvement=result["held_paired_brier_improvement"],
                    bootstrap_lower=lower,information_gate_pass=int(passed),model_fits=2,native_accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking),
                  recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(path.relative_to(card)),
    "sha256":sha(path),"bytes":path.stat().st_size} for path in sorted(artifacts.rglob("*")) if path.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+chr(10))
(card/"result_review.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":plan["experiment"],"status":result["status"],"decision":result["status"],
         "result":"result_review.json","recorder_id":result["recorder_id"],"experiment_id":result["experiment_id"],
         "recorder_uri":result["recorder_uri"],"recorder_local_dir":result["recorder_local_dir"],
         "model_fits":2,"native_accounts":0,"qualified_forward_days":0,"objective_achieved":False,
         "artifact_policy":plan["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card/"conclusion.md").write_text("# 中文结论\n\n固定 OCC 看跌占比异常相对既有 SOXX 价格、波动和成交量基线的五日下行风险信息筛选：2025 训练 "+str(len(train))+" 日，2026 已复用诊断 "+str(len(held))+" 日；两次逻辑回归拟合、一次区块 bootstrap、零账户。候选系数 "+f"{coefficient:.6f}"+"，持有期配对 Brier 改善 "+f"{result['held_paired_brier_improvement']:.8f}"+"，下界 "+f"{lower:.8f}"+"；上下半年分别 "+f"{h1.mean():.8f}"+" 和 "+f"{h2.mean():.8f}"+"。信息门 "+("通过" if passed else "失败")+"。Recorder `"+result["recorder_id"]+"`。\n\n原 OCC 历史下载不具备当年首次公开版本证明，2026 也属复用历史；无论信息门结果如何，本卡不获晋升、没有费用后 SOXX 账户或独立前向天数。失败固定版本不改阈值/符号/窗口，后续只能根据独立来源资格与新机制继续。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","plan_sha256":sha(plan_path),"recorder_id":result["recorder_id"]})+chr(10))
print(json.dumps({"status":result["status"],"train":len(train),"held":len(held),"events":held_events,
                  "paired_improvement":result["held_paired_brier_improvement"],"lower":lower,
                  "coefficient":coefficient,"recorder_id":result["recorder_id"]}))

## 中文结论

运行指标、分半年结果、Recorder 身份和晋升边界写入同卡 `conclusion.md`、`result_review.json`。历史发布版本未验证，不能从诊断性 Brier 分数推出可交易超额。